# اجرای ویترین‌یاب در Google Colab
این نوت‌بوک Laravel و FastAPI را بدون Docker اجرا می‌کند و در پایان یک لینک HTTPS موقت نمایش می‌دهد. چون مخزن Public است، به GitHub Token نیاز ندارید.

In [ ]:
# Clone the public repository
import os, shutil, subprocess
REPOSITORY = 'https://github.com/Alirezaab78/clothes_search.git'
PROJECT_DIR = '/content/clothes_search'
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, PROJECT_DIR], check=True)
print('✅ Project cloned:', PROJECT_DIR)

In [ ]:
%%bash
set -euo pipefail
PROJECT_DIR=/content/clothes_search
export DEBIAN_FRONTEND=noninteractive

# PHP, Composer, and the Laravel extensions
apt-get update -qq
apt-get install -y -qq php-cli php-curl php-mbstring php-xml php-zip php-sqlite3 unzip curl git
if ! command -v composer >/dev/null; then
  curl -fsSL https://getcomposer.org/installer -o /tmp/composer-setup.php
  php /tmp/composer-setup.php --install-dir=/usr/local/bin --filename=composer --quiet
fi

# Local Python environment and the AI service
cd $PROJECT_DIR/ai-service
python3 -m venv .venv
. .venv/bin/activate
python -m pip install --upgrade pip -q
pip install -r requirements.txt -q
nohup .venv/bin/uvicorn app.main:app --host 127.0.0.1 --port 8001 >/tmp/fashion-ai.log 2>&1 &

# Laravel and its SQLite database
cd $PROJECT_DIR/laravel-app
composer install --no-interaction --prefer-dist --optimize-autoloader
cp -n .env.example .env || true
touch database/database.sqlite
sed -i 's|^DB_CONNECTION=.*|DB_CONNECTION=sqlite|' .env
sed -i 's|^FASHION_AI_URL=.*|FASHION_AI_URL=http://127.0.0.1:8001|' .env
grep -q '^FASHION_AI_URL=' .env || echo 'FASHION_AI_URL=http://127.0.0.1:8001' >> .env
php artisan key:generate --force
php artisan migrate --force
php artisan storage:link || true
nohup php artisan serve --host=127.0.0.1 --port=8000 >/tmp/laravel.log 2>&1 &

# Wait for both local servers
for i in $(seq 1 150); do curl -fs http://127.0.0.1:8001/health >/dev/null && break; sleep 2; done
curl -fs http://127.0.0.1:8001/health
for i in $(seq 1 30); do curl -fs http://127.0.0.1:8000 >/dev/null && break; sleep 1; done
curl -fs http://127.0.0.1:8000 >/dev/null
echo '✅ Laravel and FastAPI are running.'


In [ ]:
%%bash
set -euo pipefail
curl -fsSL --retry 3 -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
nohup cloudflared tunnel --url http://127.0.0.1:8000 >/tmp/cloudflared.log 2>&1 &
for i in $(seq 1 30); do
  URL=$(grep -oE 'https://[-a-z0-9]+\.trycloudflare\.com' /tmp/cloudflared.log | head -n 1 || true)
  [ -n "$URL" ] && break
  sleep 2
done
if [ -z "${URL:-}" ]; then
  echo 'Tunnel URL was not created. Cloudflared log:'
  cat /tmp/cloudflared.log
  exit 1
fi
echo '🎉 Public URL (temporary):'
echo "$URL"


## استفاده
در یک Runtime تازه، سه سلول کد را به ترتیب یا با Runtime → Run all اجرا کنید. لینک `trycloudflare.com` در خروجی سلول آخر نشان داده می‌شود. این لینک با پایان Runtime نامعتبر می‌شود.